# CFM301 Project 1: Is the Canadian Home Bias Mean-Variance Efficient?
We trace the long-only mean-variance frontier over five regional equity ETFs (CAD total returns, Oct 2003-Aug 2026) and ask what domestic weight a mean-variance optimizer chooses, versus the ~50% Canadians actually hold and Canada's ~3% world market-cap weight.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from src.frontier import (load_returns, estimate_inputs, efficient_frontier,
                          min_variance_portfolio, max_utility_portfolio, portfolio_stats)
%matplotlib inline

## 1. Data
Monthly CAD total returns (Yahoo Finance adjusted closes; USD ETFs converted via USD/CAD). This reads only the trimmed `data/monthly_returns_cad.csv` checked into the repo.

In [ ]:
rets = load_returns()
print(rets.shape)
rets.head(3)
print(rets.index[0].date(), "to", rets.index[-1].date())

## 2. Estimated inputs
Sample means (×12), volatilities (×sqrt(12)) and the correlation matrix.

In [ ]:
est = estimate_inputs(rets)
mu, cov = est["mu"], est["cov"]
pd.DataFrame({"Mean (ann.)": mu, "Stdev (ann.)": est["vol"]}).round(4)
est["corr"].round(2)

## 3. Efficient frontier
Quadratic programs: min w′Σw s.t. w′μ = m*, Σw = 1, w ≥ 0.

In [ ]:
front = efficient_frontier(mu, cov)
w_mv = min_variance_portfolio(mu, cov)
w_opt = max_utility_portfolio(mu, cov, gamma=3)
s_mv, s_opt = portfolio_stats(w_mv, mu, cov), portfolio_stats(w_opt, mu, cov)
fig, ax = plt.subplots(figsize=(8, 5.5))
ax.plot(front["vol"], front["mu"], lw=2, label="Efficient frontier (long-only)")
ax.scatter(est["vol"], mu, s=60, zorder=5)
for a in mu.index: ax.annotate(a, (est["vol"][a], mu[a]), xytext=(6,4), textcoords="offset points")
ax.scatter([s_mv["vol"]],[s_mv["mu"]], s=90, marker="s", label="Minimum-variance")
ax.scatter([s_opt["vol"]],[s_opt["mu"]], s=110, marker="*", label="Max utility (g=3)")
ax.set_xlabel("Std dev (ann.)"); ax.set_ylabel("Expected return (ann.)"); ax.legend(); plt.show()

## 4. Selection criterion
No risk-free asset exists, so there is no tangency portfolio. We maximize mean-variance utility U = E[rp] - (g/2)Var[rp] with g = 3 (course convention), long-only.

In [ ]:
w = pd.DataFrame({"Min-Variance": w_mv, "Max utility (g=3)": w_opt})
print(w.round(4).to_string())
print(f"\nOptimal: E[r]={s_opt['mu']:.4f}, sigma={s_opt['vol']:.4f}")
print(f"Min-var: E[r]={s_mv['mu']:.4f}, sigma={s_mv['vol']:.4f}")

## 5. Sensitivity to risk aversion

In [ ]:
rows=[]
for g in [1,2,3,5,10]:
    wg = max_utility_portfolio(mu, cov, gamma=g); sg = portfolio_stats(wg, mu, cov)
    rows.append({"g":g,"E[r]":round(sg["mu"],4),"sigma":round(sg["vol"],4),**{a:round(x,3) for a,x in zip(mu.index,wg)}})
pd.DataFrame(rows)

## 6. Conclusion
At g=3 the optimizer holds **48% Canada / 52% US large cap** -- essentially the ~50% home bias Canadians exhibit. The bias looks roughly mean-variance efficient for moderate risk aversion (optimal domestic weight 28%-64% for g=2..5). Caveats: corner solution (three assets get zero weight -- error maximization), criterion-dependence (min-variance implies 89% domestic), unhedged currency, no taxes/fees.